# 02 - YOLOv8 / YOLOv8-P2 Training on ACNE04

Trains a standard YOLOv8s baseline and a custom YOLOv8s-P2 (high-resolution P2 detection head) variant on ACNE04, then evaluates and compares both against YOLOv5 and Faster R-CNN.

## Setup

In [ ]:
!git clone https://github.com/kaizar-rang/yanglab-acne.git
%cd yanglab-acne
%pip install roboflow ultralytics -q
print("\nDone.")

## Credentials & Data Download (YOLOv8 format)

In [ ]:
import os
import getpass
from pathlib import Path
from roboflow import Roboflow

if not os.path.exists("/kaggle/working/yanglab-acne"):
    !git clone https://github.com/kaizar-rang/yanglab-acne.git
%cd /kaggle/working/yanglab-acne

roboflow_key = getpass.getpass("Enter ROBOFLOW_API_KEY: ")
os.environ["ROBOFLOW_API_KEY"] = roboflow_key

rf = Roboflow(api_key=roboflow_key)
project = rf.workspace("acne-vulgaris-detection").project("acne04-detection")
dataset = project.version(1).download("yolov8", location="data/acne04_yolov8")

print("data.yaml path:", dataset.location + "/data.yaml")

## Environment Check

In [ ]:
import torch
from ultralytics import YOLO

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

data_yaml = Path("data/acne04_yolov8/data.yaml")
print(f"\ndata.yaml found: {data_yaml.exists()}")

## Part A: YOLOv8s Baseline Training

Standard YOLOv8s with pretrained COCO weights, anchor-free head, default architecture.

In [ ]:
import os
os.makedirs("runs/detect/outputs/yolov8/acne04", exist_ok=True)

model = YOLO("yolov8s.pt")

results = model.train(
    data="data/acne04_yolov8/data.yaml",
    epochs=150,
    imgsz=640,
    batch=16,
    optimizer="SGD",
    lr0=0.01,
    momentum=0.9,
    weight_decay=0.0005,
    close_mosaic=15,
    project="runs/detect",
    name="outputs/yolov8/acne04",
    exist_ok=True,
    verbose=True
)

print("\nTraining complete.")

### YOLOv8s Evaluation

In [ ]:
model = YOLO("runs/detect/outputs/yolov8/acne04/weights/best.pt")

metrics_v8 = model.val(data="data/acne04_yolov8/data.yaml", imgsz=640, split="val", plots=True)

print(f"mAP@0.5:      {metrics_v8.box.map50:.3f}")
print(f"mAP@0.5:0.95: {metrics_v8.box.map:.3f}")
print(f"Precision:    {metrics_v8.box.mp:.3f}")
print(f"Recall:       {metrics_v8.box.mr:.3f}\n")

print("Per-class mAP@0.5:")
for i, name in metrics_v8.names.items():
    print(f"  {name}: {metrics_v8.box.ap50[i]:.3f}")

## Part B: YOLOv8s-P2 (Custom High-Resolution Head)

Adds a stride-4 P2 detection head for improved sensitivity to microscopic ACNE04 lesions. Architecture defined via a custom YAML; backbone weights transfer from `yolov8s.pt` where shapes match.

In [ ]:
p2_yaml = """
# YOLOv8s with P2 (stride 4) detection head for small object detection
nc: 4  # number of classes (ACNE04: 4 lesion types)
scales:
  s: [0.33, 0.50, 1024]

backbone:
  - [-1, 1, Conv, [64, 3, 2]]    # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]   # 1-P2/4
  - [-1, 3, C2f, [128, True]]    # 2
  - [-1, 1, Conv, [256, 3, 2]]   # 3-P3/8
  - [-1, 6, C2f, [256, True]]    # 4
  - [-1, 1, Conv, [512, 3, 2]]   # 5-P4/16
  - [-1, 6, C2f, [512, True]]    # 6
  - [-1, 1, Conv, [1024, 3, 2]]  # 7-P5/32
  - [-1, 3, C2f, [1024, True]]   # 8
  - [-1, 1, SPPF, [1024, 5]]     # 9

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]          # 12 (P4)

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]          # 15 (P3)

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 2], 1, Concat, [1]]
  - [-1, 3, C2f, [128]]          # 18 (P2) -- new high-res head

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 15], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]          # 21 (P3 out)

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]          # 24 (P4 out)

  - [-1, 1, Conv, [1024, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 3, C2f, [1024]]         # 27 (P5 out)

  - [[18, 21, 24, 27], 1, Detect, [nc]]  # P2, P3, P4, P5 outputs
"""

with open("yolov8s-p2.yaml", "w") as f:
    f.write(p2_yaml)

print("Saved yolov8s-p2.yaml")

In [ ]:
os.makedirs("runs/detect/outputs/yolov8_p2/acne04", exist_ok=True)

model_p2 = YOLO("yolov8s-p2.yaml").load("yolov8s.pt")  # transfer compatible pretrained backbone weights

results_p2 = model_p2.train(
    data="data/acne04_yolov8/data.yaml",
    epochs=150,
    imgsz=640,
    batch=16,
    optimizer="SGD",
    lr0=0.01,
    momentum=0.9,
    weight_decay=0.0005,
    close_mosaic=15,
    project="runs/detect",
    name="outputs/yolov8_p2/acne04",
    exist_ok=True,
    verbose=True
)

print("\nTraining complete.")

### YOLOv8s-P2 Evaluation & Visualizations

In [ ]:
import matplotlib.pyplot as plt
import random, cv2

model_p2 = YOLO("runs/detect/outputs/yolov8_p2/acne04/weights/best.pt")

metrics_p2 = model_p2.val(data="data/acne04_yolov8/data.yaml", imgsz=640, split="val", plots=True)
save_dir = str(metrics_p2.save_dir)
print(f"Plots saved to: {save_dir}\n")

print(f"mAP@0.5:      {metrics_p2.box.map50:.3f}")
print(f"mAP@0.5:0.95: {metrics_p2.box.map:.3f}")
print(f"Precision:    {metrics_p2.box.mp:.3f}")
print(f"Recall:       {metrics_p2.box.mr:.3f}\n")

print("Per-class mAP@0.5:")
for i, name in metrics_p2.names.items():
    print(f"  {name}: {metrics_p2.box.ap50[i]:.3f}")

In [ ]:
# Confusion matrix + PR curve
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
axes[0].imshow(plt.imread(f"{save_dir}/confusion_matrix_normalized.png"))
axes[0].axis("off")
axes[0].set_title("YOLOv8-P2 Normalized Confusion Matrix")

axes[1].imshow(plt.imread(f"{save_dir}/BoxPR_curve.png"))
axes[1].axis("off")
axes[1].set_title("YOLOv8-P2 Precision-Recall Curve")

plt.tight_layout()
plt.savefig(f"{save_dir}/eval_plots.png", dpi=150)
plt.show()

In [ ]:
# Sample predictions
val_images = list(Path("data/acne04_yolov8/valid/images").glob("*.jpg"))
sample = random.sample(val_images, 6)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for ax, img_path in zip(axes, sample):
    result = model_p2(str(img_path), verbose=False)[0]
    annotated = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
    ax.imshow(annotated)
    ax.axis("off")
    ax.set_title(img_path.name, fontsize=8)

plt.suptitle("YOLOv8-P2 Predictions on ACNE04 Validation Set", fontsize=13)
plt.tight_layout()
plt.savefig(f"{save_dir}/sample_predictions.png", dpi=150)
plt.show()

In [ ]:
# Per-class mAP bar chart
class_names = list(metrics_p2.names.values())
class_map50 = metrics_p2.box.ap50

plt.figure(figsize=(8, 5))
plt.bar(class_names, class_map50, color="#378ADD")
plt.ylabel("mAP@0.5")
plt.title("YOLOv8-P2 Per-Class mAP@0.5")
plt.ylim(0, max(class_map50) * 1.2)
plt.xticks(rotation=20, ha="right")
for i, v in enumerate(class_map50):
    plt.text(i, v + 0.005, f"{v:.3f}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(f"{save_dir}/per_class_map.png", dpi=150)
plt.show()

## Final Comparison: YOLOv5 vs YOLOv8s vs YOLOv8-P2

| Class | YOLOv5 | YOLOv8s | YOLOv8-P2 |
|---|---|---|---|
| Nodules & Cysts | 0.133 | 0.079 | 0.121 |
| Papules | 0.129 | 0.082 | 0.106 |
| Pustules | 0.281 | 0.246 | 0.270 |
| Whitehead/Blackhead | 0.371 | 0.284 | 0.263 |
| **Overall mAP@0.5** | **0.229** | **0.173** | **0.190** |
| **Overall mAP@0.5:0.95** | **0.057** | **0.047** | **0.0494** |

YOLOv8-P2 outperforms the YOLOv8s baseline on every class, supporting the high-resolution head hypothesis. Both YOLOv8 variants fall short of YOLOv5, likely due to incomplete pretrained weight transfer (219/437 tensors for P2) and plateauing convergence by ~epoch 45 of the 150-epoch budget.

## Save & Download Results

In [ ]:
import shutil

!zip -r yolov8_results.zip \
    runs/detect/outputs/yolov8/acne04/weights/best.pt \
    runs/detect/outputs/yolov8/acne04/results.csv \
    runs/detect/outputs/yolov8_p2/acne04/weights/best.pt \
    {save_dir}/eval_plots.png \
    {save_dir}/sample_predictions.png \
    {save_dir}/per_class_map.png

print("Saved yolov8_results.zip")